In [ ]:
# ============================================================================
# TASK 4: FIRST-TIME HISTORICAL TRAINING TABLE
# ============================================================================
import hashlib
import io
import json
import os
import warnings

import boto3
import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')

print('=' * 80)
print('TASK 4: FIRST-TIME HISTORICAL TRAINING TABLE')
print('No production decision logs required. Synthetic cold-start slates are generated.')
print('Active action dimensions: channel, day, offer, time. Ignored: frequency, creative.')
print('=' * 80)


def get_use_case_id(default: str = 'pl-aip-uplift') -> str:
    value = os.getenv('USE_CASE_ID', default).strip()
    if not value:
        raise ValueError('USE_CASE_ID must be non-empty.')
    return value


def read_parquet_s3(bucket: str, key: str) -> pd.DataFrame:
    obj = s3_client.get_object(Bucket=bucket, Key=key)
    return pd.read_parquet(io.BytesIO(obj['Body'].read()))


def write_json_s3(payload: dict, bucket: str, key: str) -> None:
    s3_client.put_object(
        Bucket=bucket,
        Key=key,
        Body=json.dumps(payload, indent=2, default=str).encode('utf-8'),
        ContentType='application/json',
    )


def to_utc_naive(series) -> pd.Series:
    return pd.to_datetime(series, errors='coerce', utc=True).dt.tz_convert(None)


def require_columns(df: pd.DataFrame, columns: list[str], name: str) -> None:
    missing = [c for c in columns if c not in df.columns]
    if missing:
        raise ValueError(f'{name} missing required columns: {missing}')


USE_CASE_ID = get_use_case_id('pl-aip-uplift')
S3_CONFIG_BUCKET = os.getenv('S3_CONFIG_BUCKET', 'aks-nvtabular-data')
HISTORICAL_SLATE_SIZE = int(os.getenv('HISTORICAL_SLATE_SIZE', '60'))
ACTION_DIMENSION_COLS = ['channel', 'day', 'offer', 'time']
IGNORED_ACTION_DIMENSION_COLS = ['frequency', 'creative']
s3_client = boto3.client('s3')

print('[1/4] Loading action library and Task 3 reward labels...')
action_lib = read_parquet_s3(S3_CONFIG_BUCKET, f'action_library/action_library_{USE_CASE_ID}.parquet')
require_columns(action_lib, ['action_id'] + ACTION_DIMENSION_COLS, 'action_library')
action_lib = action_lib.copy()
action_lib['action_id'] = action_lib['action_id'].astype(int)
for col in ACTION_DIMENSION_COLS + [c for c in IGNORED_ACTION_DIMENSION_COLS if c in action_lib.columns]:
    action_lib[col] = action_lib[col].astype(str).str.strip()
if action_lib['action_id'].duplicated().any():
    dupes = action_lib.loc[action_lib['action_id'].duplicated(keep=False), ['action_id'] + ACTION_DIMENSION_COLS].head(10)
    raise ValueError('Action library action_id must be unique. Examples:\n' + dupes.to_string(index=False))
reward_key = f'rewards/reward_labels_{USE_CASE_ID}_LOG_SCALED.parquet'
reward_labels = read_parquet_s3(S3_CONFIG_BUCKET, reward_key)
require_columns(reward_labels, ['campaign_id', 'master_user_id', 'campaign_timestamp', 'channel_id', 'vw_cost'], 'reward_labels')
reward_labels = reward_labels.copy()
reward_labels['master_user_id'] = reward_labels['master_user_id'].astype(str)
reward_labels['campaign_id'] = reward_labels['campaign_id'].astype(str)
reward_labels['campaign_timestamp'] = to_utc_naive(reward_labels['campaign_timestamp'])
reward_labels['vw_cost'] = pd.to_numeric(reward_labels['vw_cost'], errors='coerce')
if reward_labels['campaign_timestamp'].isna().any():
    raise ValueError('reward_labels contains invalid campaign_timestamp values.')
if not reward_labels['vw_cost'].between(0, 1, inclusive='both').all():
    bad = reward_labels.loc[~reward_labels['vw_cost'].between(0, 1, inclusive='both'), ['campaign_id', 'vw_cost']].head(10)
    raise ValueError('reward_labels vw_cost must be in [0,1]. Examples:\n' + bad.to_string(index=False))
print(f'  Reward rows: {len(reward_labels):,}')

print('[2/4] Mapping historical sends to action IDs...')
action_values = {col: set(action_lib[col].astype(str)) for col in ACTION_DIMENSION_COLS}
default_dims = {col: os.getenv(f'DEFAULT_{col.upper()}', str(action_lib[col].iloc[0])) for col in ACTION_DIMENSION_COLS}
channel_alias = {
    'WHATSAPP': 'CH001', 'WA': 'CH001', 'CH001': 'CH001',
    'SMS': 'CH002', 'CH002': 'CH002',
    'RCS': 'CH003', 'GRCS': 'CH003', 'CH003': 'CH003',
    'PUSH': 'CH004', 'BPN': 'CH004', 'APN': 'CH004', 'CH004': 'CH004',
    'EMAIL': 'CH005', 'CH005': 'CH005',
}
day_alias = {
    'MON': 'DW001', 'MONDAY': 'DW001', 'DW001': 'DW001',
    'TUE': 'DW002', 'TUESDAY': 'DW002', 'DW002': 'DW002',
    'WED': 'DW003', 'WEDNESDAY': 'DW003', 'DW003': 'DW003',
    'THU': 'DW004', 'THURSDAY': 'DW004', 'DW004': 'DW004',
    'FRI': 'DW005', 'FRIDAY': 'DW005', 'DW005': 'DW005',
    'SAT': 'DW006', 'SATURDAY': 'DW006', 'DW006': 'DW006',
    'SUN': 'DW007', 'SUNDAY': 'DW007', 'DW007': 'DW007',
}


def canonicalize(col: str, value):
    if pd.notna(value):
        raw = str(value).strip()
        if raw in action_values[col]:
            return raw, False
        upper = raw.upper()
        if upper in action_values[col]:
            return upper, False
        if col == 'channel' and channel_alias.get(upper) in action_values[col]:
            return channel_alias[upper], False
        if col == 'day' and day_alias.get(upper) in action_values[col]:
            return day_alias[upper], False
    return str(default_dims[col]), True


imputed_counts = {col: 0 for col in ACTION_DIMENSION_COLS}
source_cols = {
    'channel': ['channel', 'channel_id'],
    'day': ['day'],
    'offer': ['offer'],
    'time': ['time'],
}
for col in ACTION_DIMENSION_COLS:
    values = []
    for _, row in reward_labels.iterrows():
        raw = None
        for candidate in source_cols[col]:
            if candidate in reward_labels.columns and pd.notna(row.get(candidate)):
                raw = row.get(candidate)
                break
        canon, imputed = canonicalize(col, raw)
        values.append(canon)
        imputed_counts[col] += int(imputed)
    reward_labels[col] = values

action_lookup = (
    action_lib.sort_values('action_id')
    .drop_duplicates(ACTION_DIMENSION_COLS, keep='first')
    [['action_id'] + ACTION_DIMENSION_COLS]
)
if len(action_lookup) < HISTORICAL_SLATE_SIZE:
    raise ValueError(
        f'Reduced action space has {len(action_lookup)} canonical actions for '
        f'{ACTION_DIMENSION_COLS}, but slate size is {HISTORICAL_SLATE_SIZE}. '
        'Lower HISTORICAL_SLATE_SIZE or add more valid channel/day/offer/time combinations.'
    )
mapped = reward_labels.merge(action_lookup, on=ACTION_DIMENSION_COLS, how='left', validate='many_to_one')
if mapped['action_id'].isna().any():
    sample = mapped.loc[mapped['action_id'].isna(), ['campaign_id', 'master_user_id'] + ACTION_DIMENSION_COLS].head(10)
    raise ValueError('Could not map historical rows to action_id. Examples:\n' + sample.to_string(index=False))
mapped['action_id'] = mapped['action_id'].astype(int)

print('[3/4] Generating cold-start candidate slates and synthetic propensities...')
all_action_ids = action_lookup['action_id'].astype(int).tolist()


def stable_seed(value: str) -> int:
    digest = hashlib.sha256(value.encode('utf-8')).digest()
    return int.from_bytes(digest[:8], 'little', signed=False)


def build_slate(decision_id: str, action_id: int) -> list[int]:
    others = [aid for aid in all_action_ids if aid != action_id]
    rng = np.random.default_rng(stable_seed(decision_id))
    sampled = rng.choice(others, size=HISTORICAL_SLATE_SIZE - 1, replace=False).astype(int).tolist()
    return [int(action_id)] + sampled


if 'customer_id' not in mapped.columns:
    mapped['customer_id'] = mapped['master_user_id']
base_decision = (
    mapped['campaign_id'].astype(str) + '::' +
    mapped['master_user_id'].astype(str) + '::' +
    mapped['campaign_timestamp'].dt.strftime('%Y%m%dT%H%M%S')
)
mapped['decision_id'] = ['hist::' + v + f'::{i:08d}' for i, v in enumerate(base_decision)]
mapped['timestamp'] = mapped['campaign_timestamp']
mapped['candidate_slate_json'] = [json.dumps(build_slate(did, aid)) for did, aid in zip(mapped['decision_id'], mapped['action_id'])]
mapped['slate_size'] = HISTORICAL_SLATE_SIZE
mapped['chosen_action_prob'] = 1.0 / float(HISTORICAL_SLATE_SIZE)
mapped['recommended_action_id'] = mapped['action_id']
mapped['model_run_id'] = 'historical_first_run'
mapped['decision_policy'] = 'cold_start_uniform'
mapped['exploration_epsilon'] = 1.0
mapped['propensity_source'] = 'synthetic_uniform_slate'

if mapped['decision_id'].duplicated().any():
    raise ValueError('Generated decision_id values are not unique.')
not_in_slate = [int(aid) not in set(json.loads(slate)) for aid, slate in zip(mapped['action_id'], mapped['candidate_slate_json'])]
if any(not_in_slate):
    raise ValueError('Internal slate generation error: selected action missing from slate.')

final_cols = [
    'decision_id', 'campaign_id', 'master_user_id', 'customer_id', 'timestamp', 'action_id', 'recommended_action_id',
    'vw_cost', 'chosen_action_prob', 'candidate_slate_json', 'slate_size',
    'channel', 'day', 'offer', 'time',
    'model_run_id', 'decision_policy', 'exploration_epsilon', 'propensity_source',
    'campaign_timestamp', 'channel_id', 'events_found_all', 'events_found_rewarding', 'events_found_positive',
    'total_event_reward', 'channel_cost', 'net_reward', 'reward_floor', 'reward_ceiling',
]
final_cols = [c for c in final_cols if c in mapped.columns]
df_final = mapped[final_cols].copy()

print('[4/4] Writing unified training table and audit...')
task4_audit = {
    'use_case_id': USE_CASE_ID,
    'mode': 'first_time_historical_cold_start',
    'reward_labels_uri': f's3://{S3_CONFIG_BUCKET}/{reward_key}',
    'rows': int(len(df_final)),
    'slate_size': int(HISTORICAL_SLATE_SIZE),
    'propensity_source': 'synthetic_uniform_slate',
    'has_true_logged_propensity': False,
    'has_non_logged_propensity': True,
    'strict_training_ready': False,
    'production_promotable': False,
    'active_action_dimensions': ACTION_DIMENSION_COLS,
    'ignored_action_dimensions': IGNORED_ACTION_DIMENSION_COLS,
    'imputed_action_feature_counts': imputed_counts,
    'unique_actions_hit': int(df_final['action_id'].nunique()),
    'canonical_action_count': int(len(action_lookup)),
    'min_vw_cost': float(df_final['vw_cost'].min()),
    'max_vw_cost': float(df_final['vw_cost'].max()),
}

out_buffer = io.BytesIO()
df_final.to_parquet(out_buffer, index=False)
out_buffer.seek(0)
training_key = f'training_data/unified_training_{USE_CASE_ID}.parquet'
s3_client.upload_fileobj(out_buffer, S3_CONFIG_BUCKET, training_key)

audit_key = f'training_data/task4_historical_training_audit_{USE_CASE_ID}.json'
write_json_s3(task4_audit, S3_CONFIG_BUCKET, audit_key)

print('=' * 80)
print('TASK 4 HISTORICAL TRAINING TABLE WRITTEN')
print(f'Rows             : {len(df_final):,}')
print(f'Unique actions   : {task4_audit["unique_actions_hit"]:,}')
print(f'Slate size       : {HISTORICAL_SLATE_SIZE}')
print(f'Propensity       : synthetic uniform 1/{HISTORICAL_SLATE_SIZE}')
print(f'Training parquet : s3://{S3_CONFIG_BUCKET}/{training_key}')
print(f'Audit JSON       : s3://{S3_CONFIG_BUCKET}/{audit_key}')
print('Production ready : False - first run has no true logged propensities')
print(df_final.head(5).to_string(index=False))
print('=' * 80)


In [ ]:
# ============================================================================
# TASK 4 VALIDATION: FIRST-TIME HISTORICAL CONTRACT
# ============================================================================
print('=' * 80)
print('TASK 4 VALIDATION: FIRST-TIME HISTORICAL CONTRACT')
print('=' * 80)

required_cols = [
    'decision_id', 'master_user_id', 'timestamp', 'action_id', 'vw_cost',
    'chosen_action_prob', 'candidate_slate_json', 'slate_size',
    'channel', 'day', 'offer', 'time', 'propensity_source', 'decision_policy',
]
missing = [c for c in required_cols if c not in df_final.columns]
if missing:
    raise ValueError(f'Task 4 output missing required columns: {missing}')
if df_final['decision_id'].isna().any() or df_final['decision_id'].duplicated().any():
    raise ValueError('decision_id must be non-null and unique in Task 4 output.')
if not df_final['chosen_action_prob'].between(0, 1, inclusive='right').all():
    raise ValueError('chosen_action_prob must be in (0,1].')
if not df_final['vw_cost'].between(0, 1, inclusive='both').all():
    raise ValueError('vw_cost must be in [0,1].')


def _contains_action(row):
    slate = json.loads(row['candidate_slate_json'])
    return int(row['action_id']) in {int(aid) for aid in slate} and len(slate) == int(row['slate_size'])


not_in_slate = ~df_final.apply(_contains_action, axis=1)
if not_in_slate.any():
    raise ValueError(f'action_id missing from candidate_slate_json for {int(not_in_slate.sum()):,} rows.')
if not (df_final['decision_policy'].astype(str) == 'cold_start_uniform').all():
    raise ValueError('First-time training rows must be marked decision_policy=cold_start_uniform.')
if not (df_final['propensity_source'].astype(str) == 'synthetic_uniform_slate').all():
    raise ValueError('First-time training rows must be marked propensity_source=synthetic_uniform_slate.')

print(f'Output rows         : {len(df_final):,}')
print(f'Unique decision ids : {df_final["decision_id"].nunique():,}')
print(f'Unique actions      : {df_final["action_id"].nunique():,}')
print(f'Chosen prob         : {df_final["chosen_action_prob"].min():.6f} - {df_final["chosen_action_prob"].max():.6f}')
print(f'VW cost range       : {df_final["vw_cost"].min():.6f} - {df_final["vw_cost"].max():.6f}')
print('VALIDATION PASSED: first-time historical training table is ready for Task 5.')
print('=' * 80)
